# Netflix Data Analysis - Exploratory Data Analysis
## Comprehensive analysis of Netflix content catalog

This notebook provides an in-depth exploration of the Netflix dataset including:
- Data quality assessment
- Statistical summaries
- Trend analysis
- Visualization creation

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")

## 1. Data Loading

In [ ]:
# Load the unified Netflix catalog
df = pd.read_csv('../data/merged/netflix_unified_catalog.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")

## 2. Data Overview

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Column information
df.info()

In [ ]:
# Statistical summary
df.describe()

## 3. Data Quality Assessment

In [ ]:
# Missing values analysis
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending=False
)

print("Missing Values Summary:")
print(missing_data)

In [ ]:
# Visualize missing data
if not missing_data.empty:
    fig = px.bar(missing_data, x='Column', y='Missing_Percentage',
                title='Missing Data by Column',
                labels={'Missing_Percentage': 'Missing %'},
                color='Missing_Percentage',
                color_continuous_scale='Reds')
    fig.show()
else:
    print("✅ No missing values in dataset!")

## 4. Content Distribution Analysis

In [ ]:
# Movies vs TV Shows
type_dist = df['type'].value_counts()
print("Content Type Distribution:")
print(type_dist)
print(f"\nMovies: {type_dist['Movie']/len(df)*100:.1f}%")
print(f"TV Shows: {type_dist['TV Show']/len(df)*100:.1f}%")

In [ ]:
# Visualize content distribution
fig = px.pie(values=type_dist.values, names=type_dist.index,
            title='Movies vs TV Shows Distribution',
            color_discrete_sequence=['#E50914', '#564d4d'])
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 5. Temporal Analysis

In [ ]:
# Content releases over time
yearly_releases = df.groupby(['release_year', 'type']).size().unstack(fill_value=0)

fig = px.area(yearly_releases, x=yearly_releases.index, y=yearly_releases.columns,
             title='Netflix Content Releases Over Time',
             labels={'value': 'Number of Releases', 'release_year': 'Year'},
             color_discrete_sequence=['#E50914', '#564d4d'])
fig.show()

In [ ]:
# Decade analysis
decade_dist = df['decade'].value_counts().sort_index()
print("Content by Decade:")
print(decade_dist)

## 6. Genre Analysis

In [ ]:
# Top 15 genres
top_genres = df['primary_genre'].value_counts().head(15)

fig = px.bar(x=top_genres.values, y=top_genres.index, orientation='h',
            title='Top 15 Genres on Netflix',
            labels={'x': 'Number of Titles', 'y': 'Genre'},
            color=top_genres.values,
            color_continuous_scale='Reds')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Genre performance metrics
genre_stats = df.groupby('primary_genre').agg({
    'vote_average': 'mean',
    'popularity': 'mean',
    'success_score': 'mean',
    'title': 'count'
}).round(2)

genre_stats.columns = ['Avg_Rating', 'Avg_Popularity', 'Success_Score', 'Count']
genre_stats = genre_stats[genre_stats['Count'] >= 10]  # Minimum 10 titles
genre_stats = genre_stats.sort_values('Success_Score', ascending=False).head(10)

print("Top 10 Performing Genres:")
print(genre_stats)

## 7. Rating Analysis

In [ ]:
# Rating distribution
fig = px.histogram(df, x='vote_average', nbins=50,
                  title='Rating Distribution',
                  labels={'vote_average': 'Rating', 'count': 'Frequency'},
                  color_discrete_sequence=['#E50914'])
fig.add_vline(x=df['vote_average'].mean(), line_dash="dash", 
             annotation_text=f"Mean: {df['vote_average'].mean():.2f}")
fig.show()

In [ ]:
# Rating by content type
fig = px.box(df, x='type', y='vote_average', color='type',
            title='Rating Distribution by Content Type',
            color_discrete_sequence=['#E50914', '#564d4d'])
fig.show()

## 8. Geographic Analysis

In [ ]:
# Top 20 producing countries
top_countries = df['primary_country'].value_counts().head(20)

fig = px.bar(x=top_countries.values, y=top_countries.index, orientation='h',
            title='Top 20 Content Producing Countries',
            labels={'x': 'Number of Titles', 'y': 'Country'},
            color=top_countries.values,
            color_continuous_scale='Reds')
fig.update_layout(showlegend=False, height=600)
fig.show()

## 9. Financial Analysis (Movies Only)

In [ ]:
# Movie financial metrics
movies = df[df['type'] == 'Movie'].copy()

print(f"Total Movies: {len(movies):,}")
print(f"Total Budget: ${movies['budget'].sum()/1e9:.2f}B")
print(f"Total Revenue: ${movies['revenue'].sum()/1e9:.2f}B")
print(f"Average ROI: {movies['roi'].mean():.1f}%")

In [ ]:
# Budget vs Revenue scatter plot
movies_clean = movies[(movies['budget'] > 0) & (movies['revenue'] > 0)]

fig = px.scatter(movies_clean, x='budget', y='revenue',
                hover_data=['title', 'release_year'],
                color='vote_average',
                size='popularity',
                title='Movie Budget vs Revenue',
                labels={'budget': 'Budget ($)', 'revenue': 'Revenue ($)'},
                color_continuous_scale='Reds')
fig.show()

## 10. Top Content Recommendations

In [ ]:
# Top 20 by success score
top_content = df.nlargest(20, 'success_score')[[
    'title', 'type', 'release_year', 'primary_genre',
    'vote_average', 'popularity', 'success_score'
]]

print("Top 20 Content by Success Score:")
print(top_content)

## 11. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['release_year', 'vote_average', 'vote_count', 'popularity',
               'success_score', 'content_age', 'genre_count', 'cast_count']

correlation_matrix = df[numeric_cols].corr()

# Visualize correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r',
           center=0, square=True, linewidths=1)
plt.title('Correlation Matrix of Netflix Content Features')
plt.tight_layout()
plt.show()

## 12. Key Insights Summary

In [ ]:
# Generate insights summary
insights = f"""
=== NETFLIX ANALYTICS - KEY INSIGHTS ==="

📊 CONTENT OVERVIEW:
- Total Content: {len(df):,} pieces
- Movies: {len(df[df['type']=='Movie']):,} ({len(df[df['type']=='Movie'])/len(df)*100:.1f}%)
- TV Shows: {len(df[df['type']=='TV Show']):,} ({len(df[df['type']=='TV Show'])/len(df)*100:.1f}%)
- Date Range: {df['release_year'].min()} - {df['release_year'].max()}

⭐ QUALITY METRICS:
- Average Rating: {df['vote_average'].mean():.2f}/10
- Highly Rated (≥7.5): {len(df[df['vote_average']>=7.5]):,} ({len(df[df['vote_average']>=7.5])/len(df)*100:.1f}%)
- Top Rated: {df.nlargest(1, 'vote_average')['title'].iloc[0]} ({df.nlargest(1, 'vote_average')['vote_average'].iloc[0]:.2f})

🎭 GENRE INSIGHTS:
- Total Genres: {df['primary_genre'].nunique()}
- Most Common: {df['primary_genre'].value_counts().index[0]} ({df['primary_genre'].value_counts().iloc[0]:,} titles)
- Best Performing: {genre_stats.index[0]} (Score: {genre_stats['Success_Score'].iloc[0]:.2f})

🌍 GEOGRAPHIC INSIGHTS:
- Countries Represented: {df['primary_country'].nunique()}
- Top Producer: {df['primary_country'].value_counts().index[0]} ({df['primary_country'].value_counts().iloc[0]:,} titles)

💰 FINANCIAL INSIGHTS (Movies):
- Total Budget: ${movies['budget'].sum()/1e9:.2f}B
- Total Revenue: ${movies['revenue'].sum()/1e9:.2f}B
- Average ROI: {movies['roi'].mean():.0f}%
- Highest Revenue: {movies.nlargest(1, 'revenue')['title'].iloc[0]} (${movies.nlargest(1, 'revenue')['revenue'].iloc[0]/1e6:.0f}M)
"""

print(insights)

## Next Steps

1. **Deep Dive Analysis**: Pick specific genres or time periods for detailed study
2. **Predictive Modeling**: Build models to predict content success
3. **Recommendation System**: Create content recommendation algorithms
4. **Sentiment Analysis**: Analyze content descriptions
5. **Dashboard**: View interactive dashboard with `streamlit run app/streamlit_app.py`